In [ ]:
from google.colab import drive
drive.mount(mountpoint="/content/drive/",force_remount=True)

Mounted at /content/drive/


In [ ]:
!pip install mem0ai langchain-community langchain-openai qdrant-client langgraph langchain-qdrant anthropic pymupdf ragas datasets && pip install --no-deps requests==2.32.4


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2


In [ ]:
from google.colab import userdata
ANTHROPIC_KEY=userdata.get("ANTHROPIC_KEY")
GEMINI_KEY=userdata.get("GEMINI_KEY")
OPEN_AI_KEY=userdata.get("OPEN_AI_KEY")
QDRANT_KEY=userdata.get("QDRANT_KEY")
QDRANT_URL=userdata.get("QDRANT_URL")

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct,VectorParams,Distance
from pathlib import Path
from openai import OpenAI
import pymupdf
import json
import os
import pickle
import numpy
import pandas
import uuid

In [ ]:
qclient=QdrantClient(
    api_key=QDRANT_KEY,
    url=QDRANT_URL
)

In [ ]:
oai=OpenAI(api_key=OPEN_AI_KEY)

In [ ]:
CHUNK_SIZE1=1000
CHUNK_OVERLAP=400
COLLECTION_NAME="prodrag"
EMBEDDING_MODEL="text-embedding-3-large"
INFERENCE_MODEL= "gpt-5.5"

In [ ]:
#!STEPS
#!CHUNKING MODEL
#!EMBEDDING DEFINE AND EMBED CHUNKS
#!PROPERLY SAVE IN QDRANT
#!EMBED QUERY
#!RETRIEVE
#!BUILD CONTEXT
#!ANSWER


In [ ]:
def chunk_text(raw_content:str,chunk_size:int=1000,chunk_overlap:int=200):
  start=0
  chunks_list=[]
  while start<len(raw_content):
    end=start+chunk_size
    chunks_list.append(raw_content[start:end])
    start+=chunk_size-chunk_overlap
  return chunks_list


In [ ]:

def embed_text(text_data: str | list[str]) -> list[list[float]]:
    embedded_text = oai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text_data
    )

    return [
        node.embedding
        for node in sorted(
            embedded_text.data,
            key=lambda node: node.index
        )
    ]


In [ ]:
def file_reader(file_path: str) -> str:
    ext = Path(file_path).suffix.lower()
    content_str = ""

    if ext == ".pdf":
        with pymupdf.open(file_path) as doc:
            for page in doc:
                content_str += page.get_text()

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            content_str = f.read()

    else:
        content_str = "Invalid file format"

    return content_str


In [ ]:

def ensure_collection(collection_name:str):
  collection_list=qclient.get_collections().collections
  exists=any(collection_name==collection.name for collection in collection_list)
  if not exists:
    qclient.create_collection(
        collection_name=collection_name,
        vector_config=VectorParams(
            size=3072,
            distance=Distance.COSINE
        )
    )

In [ ]:



def index_file(file_path: str, collection_name: str):
    text_content = file_reader(file_path)
    chunked_data = chunk_text(raw_content=text_content)

    if not chunked_data:
        return 0

    # Generate one embedding per chunk
    embedded_data = embed_text(text_data=chunked_data)

    if len(chunked_data) != len(embedded_data):
        raise ValueError(
            "Number of chunks and embeddings must match"
        )

    points = []

    for chunk, embedding in zip(chunked_data, embedded_data):
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload={
                    "text": chunk,
                    "source": str(file_path)
                }
            )
        )

    qclient.upsert(
        collection_name=collection_name,
        points=points,
        wait=True
    )

    return len(points)


In [ ]:
pdf_path="/content/drive/MyDrive/ml_repo/pdf_data/kalibangan.pdf"
files=[pdf_path]
for file in files:
  index_file(file,COLLECTION_NAME)

In [ ]:
def retrieve_and_answer(user_query:str):
  embedding_query=oai.embeddings.create(
      model=EMBEDDING_MODEL,
      input=user_query
  )
  query_vector=embedding_query.data[0].embedding
  results=qclient.query_points(
      collection_name=COLLECTION_NAME,
      query=query_vector,
      limit=5
  ).points
  context=""
  for i,result in enumerate(results):
    context+=f"""
    {result.payload['text']}
    {result.payload['source']}
    {result.score}\n\n"""
  context_message=f"YOU ARE AN EXPERT SYSTEM THAT ANSWERS WITH REFERENCE AND OWN KNOWLEDGE AND GIVE REFERENCE{context}"
  llm_answer=oai.chat.completions.create(
      model=INFERENCE_MODEL,
      messages=[{
          "role":"system","content":context_message
      },
      {
          "role":"user","content":user_query
      }
        ]
  )
  return {
      "answer":llm_answer.choices[0].message.content,
      "context":context
      }



In [ ]:
retrieve_and_answer("How was kalibangan compared to other sites?")

{'answer': 'Kalibangan was a **medium-sized Harappan town**, not one of the largest Indus cities, but larger and more important than small rural settlements.\n\n- **Size:** Kalibangan covered about **11.5 hectares** (Possehl 1999:773).  \n- **Compared with major cities:** It was much smaller than **Rakhigarhi** (~124 ha) and **Harappa** (~150 ha). It lay roughly between them—about **200 km from Rakhigarhi** and **175 km from Harappa** (Nath 2014:306; Possehl 2002:66).  \n- **Compared with villages/farmsteads:** It was larger than nearby smaller sites such as **Tarkhanwala Dera**, which was about **3 hectares** (Ghosh 1989).  \n- **Importance:** Despite its smaller size, Kalibangan was significant because it had two settlement areas, Early and Mature Harappan occupation, one of the earliest known **ploughed fields**, and ritual features such as **fire altars**, comparable to examples at **Banawali** and **Lothal** (Lal et al. 2015a:6).  \n\nSo, Kalibangan can be described as a **regiona

In [ ]:
retrieve_and_answer("fire altar in kalibangan?")

{'answer': 'Yes. **Kalibangan**, a Harappan/Indus site in Rajasthan on the Ghaggar river bed, is famous for the discovery of **fire altars**.\n\nKey points:\n\n- In the **Mature Harappan phase** of Kalibangan, excavators found **seven fire hearths/altars** in a row in a non-residential area, especially associated with the **citadel area**.  \n- These were interpreted as **ritualistic fire altars**, not ordinary domestic hearths, because of their arrangement and context.  \n- Similar fire-altars or ritual hearths have also been reported from **Banawali** and **Lothal**.  \n- The finds usually included burnt material such as **ash, charcoal, burnt clay/terracotta cakes**, and brick-lined or prepared hearth structures.  \n- Scholars generally treat them as evidence of **ritual fire practices** in the Indus tradition, but they should not automatically be identified with later Vedic yajña without caution.\n\n**References:**  \n- Lal, B. B. et al. 2015. *Kalibangan excavation reports*, Archa

In [ ]:
#!LANGGRAPH AND MEM0 computation

In [ ]:
#!langchain rag

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
pdf_loader=PyMuPDFLoader(file_path="/content/drive/MyDrive/ml_repo/pdf_data/OCP1.pdf")

In [ ]:
docs=pdf_loader.load()

In [ ]:
docs[0]

Document(metadata={'producer': 'Nitro Pro 9  (9. 5. 1. 5)', 'creator': 'Nitro Pro 9  (9. 5. 1. 5)', 'creationdate': '2020-03-29T08:16:22+00:00', 'source': '/content/drive/MyDrive/ml_repo/pdf_data/OCP1.pdf', 'file_path': '/content/drive/MyDrive/ml_repo/pdf_data/OCP1.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-03-29T13:46:28+05:30', 'trapped': '', 'modDate': "D:20200329134628+05'30'", 'creationDate': 'D:20200329081622Z', 'page': 0}, page_content='3/29/2020\n1\nxSfjd e`n~Hkk.M\nDr. Anil Kumar\nProfessor\nAncient Indian History and Archaeology\nUniversity of Lucknow\nanilkumararch@gmail.com\nKumar_anil@lkouniv.ac.in\nOchre Coloured Pottery\n\uf0d8The term Ochre Coloured Pottery Culture (OCP) was coined by\nB.B. Lal after he excavated the site Hastinapur in the year 1951-52.\n\uf0d8This type of pottery was first discovered at Bisauli in Badaun district &\nRajpur Parsu in Bijnaur district by B.B. Lal in 1950-51.\n\uf

In [ ]:
openai_embeddings=OpenAIEmbeddings(api_key=OPEN_AI_KEY,model=EMBEDDING_MODEL)

In [ ]:
qdrant_vector_store=QdrantVectorStore(
    client=qclient,
    collection_name=COLLECTION_NAME,
    embedding=openai_embeddings
)

In [ ]:
text_splliter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
chunks=text_splliter.split_documents(documents=docs)

In [ ]:
vector_store=QdrantVectorStore.from_documents(
    collection_name=COLLECTION_NAME,
    documents=chunks,
    url=QDRANT_URL,
    api_key=QDRANT_KEY,
    embedding=openai_embeddings
)
#!CHUNKING PARTS,EMBEDDING AND STORING AT ONCE

In [ ]:
def retrieve_model_lchain(user_query:str):
  vectore_store=QdrantVectorStore.from_existing_collection(
            embedding=openai_embeddings,
            url=QDRANT_URL,
            collection_name=COLLECTION_NAME,
            api_key=QDRANT_KEY
        )
  similarity_vector=vector_store.similarity_search(query=user_query,k=5)
  result_val=[]
  for result in similarity_vector:
      result_str=f"""
                  Page: {result.metadata.get("page_label", "Unknown")}
                  Source: {result.metadata.get("source", "Unknown")}
                  Content: {result.page_content}
                  """
      result_val.append(result_str)

  context="\n\n".join(result_val)


  SYSTEM_PROMPT=f"""YOU ARE AN SMART AI ASSISTANT TO HELP IN VARIOUS FACETS
  context from pdf and you should use the context from txt and pdf,show page number and data
  {context}
  """


  response=oai.chat.completions.create(
      model="gpt-5.4",
      messages=[
          {"role":"system","content":SYSTEM_PROMPT},
          {"role":"user","content":user_query}
      ]
  )
  return {
      "answer":response.choices[0].message.content,
      "context":context}


In [ ]:
retrieve_model_lchain("How was ocp in terms of material culture?")

{'answer': 'In terms of **material culture**, the **OCP (Ochre Coloured Pottery) culture** appears to have been **fairly simple but varied**, with evidence of pottery, terracotta objects, stone tools, and in some sites copper objects.\n\n### Material culture of OCP\n\nFrom the provided text:\n\n- **Lal Qila artefacts were made of four materials**:\n  - **stone**\n  - **terracotta**\n  - **bone**\n  - **copper**  \n  **Source:** *OCP1.pdf, Page: Unknown*  \n  > “Lal Qila artefacts were made of four materials i.e. stone, terracotta, bone, and copper...”\n\n- **Atranjikhera and Daulatpur** were said to be **devoid of bone and copper artefacts**.  \n  **Source:** *OCP1.pdf, Page: Unknown*  \n  > “...while Atranjikhera and Daulatpura were devoid of artefacts made of bone and copper.”\n\n- At **Saipai**, OCP was associated with **Copper Hoards**, including:\n  - a **hooked spearhead**\n  - a **harpoon**  \n  **Source:** *OCP1.pdf, Page: Unknown*\n\n- At **Atranjikhera**, finds included:\n  -